# 03 — Fine-Tune Sweep

**Goal:** Fine-tune DistilBERT on Banking77 at varying training-set sizes. Evaluate each model on the same Banking77 test set used by the LLM baseline in notebook 02.

**Method:** 7 training sizes × 5 random seeds. Random seeds rather than k-fold CV — at n=50 with 77 classes, k-fold puts most classes at zero examples per fold, which makes the metrics meaningless. Random seeds give the same variance estimate without the degenerate-fold problem.

**Compute:** Designed to run on free Google Colab (T4 GPU). Total compute budget ~60 minutes for the full 35-run sweep (Banking77 queries are short, so each fine-tune is fast).

## Sections

1. Load training subsets + fixed val + test
2. Tokenisation + DataLoader setup
3. Fine-tuning loop — DistilBERT × 7 training sizes × 5 random seeds
4. Evaluate each model on the test set
5. Save predictions + per-run metadata to `results/`

## 1. Load training subsets + val + test

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch

sys.path.insert(0, str(Path('..').resolve()))
from src.data import load_test_set, load_val_set, load_train_subset, load_label_names, TRAINING_SIZES, N_SEEDS

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# TODO: test_df = load_test_set(); val_df = load_val_set(); label_names = load_label_names()

## 2. Tokenisation

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 64   # banking queries are short — verify Banking77 token-length distribution in notebook 01
NUM_LABELS = 77

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# TODO: tokenise val + test once (they're constant); tokenise each train subset inside the loop below

## 3. Fine-tuning loop

For each (training_size, seed), fine-tune a fresh DistilBERT from scratch and evaluate on the full 3,080-row test set. Same val set every run — makes early stopping comparable across runs.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

EPOCHS = 5  # more epochs because queries are short
BATCH_SIZE = 32  # fits comfortably on T4 with MAX_LENGTH=64
LR = 2e-5

# TODO: nested loop — for each n in TRAINING_SIZES, for each seed in 0..N_SEEDS:
#   - load_train_subset(n, seed)
#   - init fresh DistilBERT with num_labels=77 (seed the model init too — transformers `set_seed(seed)`)
#   - fine-tune; early-stop on the fixed val set
#   - evaluate on the full 3,080-row test set
#   - append predictions + metadata (training_size, seed, train_time_s, best_val_loss, gpu_type) to results

## 4. Evaluate + save results

For each (training_size, seed) combination, save:
- Per-row predictions on the test set (so notebook 04 can do the paired bootstrap)
- Training duration (wall-clock)
- Best validation loss
- Metadata (hyperparameters, GPU type, etc.)

Path: `results/finetune_predictions.parquet` (long format, columns: training_size, seed, test_row_id, true_label, pred_label)
Path: `results/finetune_metadata.parquet` (one row per (training_size, seed))

In [ ]:
# TODO

## Notes / gotchas

- *(write up things that went wrong here — they become the most valuable part of the writeup)*
- **Seed the model init too**, not just the data shuffle. `transformers.set_seed(seed)` seeds Python, numpy, torch, and CUDA in one call. Without it the DistilBERT head init is random every run and dominates the variance you're trying to measure.
- **At n=50 with 77 classes, some intents will have zero training examples** in some seeds. The model can never predict those intents correctly — expect macro-F1 to be low for reasons that aren't really about model capacity. Track and report per-(n, seed) which intents are missing.
- **Fine-tune head is randomly initialised**; first few epochs may swing wildly before settling. Log eval per epoch, take the best-val checkpoint, not the final.